# US Active vs Passive — Production Colab Runner

This project compares a real Firstrade active portfolio with cash-flow-matched passive portfolios. Repository: [hh4832/us_active_vs_passive](https://github.com/hh4832/us_active_vs_passive).

This notebook is the production Google Colab runner using the Tiingo-only production price backend. It only orchestrates repository refresh, dependency installation, authentication, input validation, tests, pipeline execution, result review, and Google Drive export. Core analysis logic remains in `src/` and `scripts/run_analysis.py`. Run all cells from a blank Colab runtime.

In [ ]:
%cd /content
!rm -rf /content/us_active_vs_passive
!git clone https://github.com/hh4832/us_active_vs_passive.git
%cd /content/us_active_vs_passive
!git log -1 --oneline
!git status


In [ ]:
!pip install -q -r requirements.txt
import platform
import requests
import numpy as np
import pandas as pd

print("Python:", platform.python_version())
print("requests:", requests.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
import os
from google.colab import userdata

try:
    tiingo_token = userdata.get("TIINGO_API_TOKEN")
except Exception as exc:
    raise RuntimeError("請在 Colab 左側 Secrets 新增 TIINGO_API_TOKEN") from exc
if not tiingo_token:
    raise RuntimeError("請在 Colab 左側 Secrets 新增 TIINGO_API_TOKEN")
os.environ["TIINGO_API_TOKEN"] = tiingo_token
del tiingo_token
print("TIINGO_API_TOKEN loaded from Colab Secrets.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Change only these parameters when the input source or mounted Drive path differs.
TRADE_INPUT_MODE = "upload"  # "upload" or "drive"
TRADE_CSV_DRIVE_PATH = "/content/drive/MyDrive/path/to/firstrade.csv"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/US_Active_vs_Passive"

# The shared folder ID is not a Linux path. Use its actual mounted MyDrive path above.
GOOGLE_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1NlsfPTDpI1hDRso6NwERT7xYvlLOe1g2?usp=drive_link"
if TRADE_INPUT_MODE not in {"upload", "drive"}:
    raise RuntimeError("TRADE_INPUT_MODE must be 'upload' or 'drive'.")

In [ ]:
import hashlib
import shutil
from pathlib import Path

import yaml
from IPython.display import display
from google.colab import files
from src.load_transactions import load_transactions
from src.normalize_transactions import normalize_transactions

INPUT_PATH = Path("data/raw/firstrade.csv")
INPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if TRADE_INPUT_MODE == "upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("請只上傳 1 個 Firstrade CSV 或 Excel 檔案。")
    original_input_filename, uploaded_bytes = next(iter(uploaded.items()))
    source_path = Path(original_input_filename)
    source_path.write_bytes(uploaded_bytes)
else:
    source_path = Path(TRADE_CSV_DRIVE_PATH)
    original_input_filename = source_path.name
    if not source_path.is_file():
        raise RuntimeError(f"找不到 Firstrade 檔案：{source_path}")

if source_path.suffix.lower() not in {".csv", ".xlsx"}:
    raise RuntimeError("Firstrade input 必須是 CSV 或 XLSX 檔案。")
input_size_bytes = source_path.stat().st_size
input_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
try:
    raw_transactions = load_transactions(source_path)
    aliases = yaml.safe_load(Path("config/config.yaml").read_text(encoding="utf-8"))["column_aliases"]
    normalized_preview, detected_columns = normalize_transactions(raw_transactions, aliases)
except Exception as exc:
    raise RuntimeError(f"Firstrade 檔案無法解析，分析已停止：{exc}") from exc

if source_path.suffix.lower() == ".csv":
    if source_path.resolve() != INPUT_PATH.resolve():
        shutil.copy2(source_path, INPUT_PATH)
else:
    raw_transactions.to_csv(INPUT_PATH, index=False)

date_values = normalized_preview["date"].dropna()
ticker_universe = sorted(normalized_preview["ticker"].dropna().astype(str).unique().tolist())
print("Original filename:", original_input_filename)
print("File size (bytes):", input_size_bytes)
print("SHA256:", input_sha256)
print("Row count:", len(raw_transactions))
print("Columns:", list(raw_transactions.columns))
print("Date range:", date_values.min() if not date_values.empty else None, "to", date_values.max() if not date_values.empty else None)
print("Ticker universe:", ticker_universe)
print("Detected columns:", detected_columns)

In [ ]:
import re
import subprocess

pytest_run = subprocess.run(["pytest", "-q"], capture_output=True, text=True)
pytest_text = (pytest_run.stdout + pytest_run.stderr).strip()
print(pytest_text)
passed_match = re.search(r"(\d+) passed", pytest_text)
tests_result = passed_match.group(0) if passed_match else "FAILED"
if pytest_run.returncode != 0 or not passed_match:
    raise RuntimeError(f"Unit tests failed; got exit={pytest_run.returncode}, result={tests_result}")

In [ ]:
pipeline_run = subprocess.run(["python", "scripts/run_analysis.py", "data/raw/firstrade.csv"], capture_output=True, text=True)
print((pipeline_run.stdout + pipeline_run.stderr).strip())
if pipeline_run.returncode != 0:
    raise RuntimeError(f"Production pipeline failed with exit code {pipeline_run.returncode}.")

In [ ]:
run_candidates = sorted(Path("outputs").glob("run_[0-9]*_[0-9]*"), key=lambda p: p.stat().st_mtime)
if not run_candidates:
    raise RuntimeError("找不到 outputs/run_YYYYMMDD_HHMMSS；production pipeline 未產生輸出。")
latest_run = run_candidates[-1]
run_timestamp = latest_run.name.removeprefix("run_")
top_level_output_folders = sorted(p.name for p in latest_run.iterdir() if p.is_dir())
print("Latest run path:", latest_run.resolve())
print("Run timestamp:", run_timestamp)
print("Top-level output folders:", top_level_output_folders)

In [ ]:
summary_path = latest_run / "summary/performance_summary.csv"
if not summary_path.is_file():
    raise RuntimeError(f"Missing required performance summary: {summary_path}")
performance = pd.read_csv(summary_path)
selected = performance.loc[performance["rf_version"].eq("rf_config")].copy()
capture_path = latest_run / "risk/capture_ratios.csv"
selected["upside_capture"] = np.nan
selected["downside_capture"] = np.nan
if capture_path.is_file():
    capture = pd.read_csv(capture_path).set_index("regime")
    selected.loc[selected["portfolio"].eq("active_equity_sleeve"), "upside_capture"] = capture.get("arithmetic_daily_capture", pd.Series(dtype=float)).get("up", np.nan)
    selected.loc[selected["portfolio"].eq("active_equity_sleeve"), "downside_capture"] = capture.get("arithmetic_daily_capture", pd.Series(dtype=float)).get("down", np.nan)
else:
    print(f"WARNING: optional capture table missing: {capture_path}")
selected["beta"] = selected.get("vs_voo_beta", np.nan)
portfolio_order = ["active_risk_assets", "active_equity_sleeve", "active_plus_sgov", "full_actual_account", "voo", "vgt", "voo_sgov_80_20", "beta_matched_voo_sgov", "volatility_matched_voo_sgov"]
summary_columns = ["portfolio", "cumulative_return", "annualized_return", "annualized_volatility", "max_drawdown", "sharpe", "sortino", "calmar", "beta", "upside_capture", "downside_capture"]
selected["portfolio"] = pd.Categorical(selected["portfolio"], portfolio_order, ordered=True)
display(selected.loc[selected["portfolio"].notna(), summary_columns].sort_values("portfolio").reset_index(drop=True))
for extra_name in ["account_reconciliation.csv"]:
    extra_path = latest_run / "summary" / extra_name
    if extra_path.is_file():
        print(f"Summary: {extra_name}")
        display(pd.read_csv(extra_path))

In [ ]:
risk_files = ["drawdown_episodes.csv", "regime_analysis.csv", "capture_ratios.csv", "beta_matched_results.csv", "rolling_metrics.csv"]
for filename in risk_files:
    path = latest_run / "risk" / filename
    if path.is_file():
        print(f"Risk table: {filename}")
        display(pd.read_csv(path).head(20))
    else:
        print(f"WARNING: optional risk table missing: {path}")

coverage_files = ["ticker_coverage_audit.csv", "trade_date_coverage_audit.csv", "holding_period_coverage.csv", "corporate_actions.csv"]
for filename in coverage_files:
    path = latest_run / "trades" / filename
    if not path.is_file():
        raise RuntimeError(f"Missing required Tiingo audit: {path}")
    print(f"Tiingo audit: {filename}")
    display(pd.read_csv(path).head(20))

In [ ]:
reconciliation_path = latest_run / "summary/account_reconciliation.csv"
if not reconciliation_path.is_file():
    raise RuntimeError(f"Missing reconciliation output: {reconciliation_path}")
reconciliation = pd.read_csv(reconciliation_path)
display(reconciliation)
daily_path = latest_run / "daily/daily_nav.csv"
if daily_path.is_file():
    daily = pd.read_csv(daily_path)
    max_abs_difference = daily["reconciliation_difference"].abs().max()
    valid_nav = daily["nav"].abs().replace(0, np.nan)
    max_pct_difference = (daily["reconciliation_difference"].abs() / valid_nav).max()
    print("Max reconciliation difference (USD):", max_abs_difference)
    print("Max reconciliation difference (%):", max_pct_difference * 100)
    if max_abs_difference > 1 or max_pct_difference > 0.0001:
        print("WARNING: reconciliation difference exceeds $1 or 0.01%.")
else:
    print(f"WARNING: daily NAV file missing; percentage reconciliation could not be checked: {daily_path}")

In [ ]:
from IPython.display import Image, display

chart_names = ["equity_curve.png", "drawdown_curve.png", "rolling_volatility.png", "rolling_beta.png", "allocation_over_time.png", "contribution.png"]
for chart_name in chart_names:
    chart_path = latest_run / "figures" / chart_name
    if chart_path.is_file():
        print(chart_name)
        display(Image(filename=str(chart_path)))
    else:
        print(f"WARNING: chart missing: {chart_path}")

In [ ]:
drive_mount_root = Path("/content/drive/MyDrive")
if not drive_mount_root.is_dir():
    raise RuntimeError("Google Drive 未正確 mount：找不到 /content/drive/MyDrive。")
drive_output_root = Path(DRIVE_OUTPUT_ROOT)
if not drive_output_root.is_dir():
    raise RuntimeError(f"DRIVE_OUTPUT_ROOT 不存在：{drive_output_root}。請填入該資料夾在 MyDrive 中的實際路徑。")
drive_run_path = drive_output_root / latest_run.name
shutil.copytree(latest_run, drive_run_path, dirs_exist_ok=True)
(drive_output_root / "latest_run.txt").write_text(latest_run.name + "\n", encoding="utf-8")
print("Google Drive output path:", drive_run_path)

In [ ]:
import json
import subprocess

run_info_path = latest_run / "metadata/run_info.json"
warnings_path = latest_run / "metadata/warnings.json"
if not run_info_path.is_file():
    raise RuntimeError(f"Missing run metadata: {run_info_path}")
run_info = json.loads(run_info_path.read_text(encoding="utf-8"))
warnings = json.loads(warnings_path.read_text(encoding="utf-8")) if warnings_path.is_file() else []
git_commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
completion_summary = {
    "git_commit_sha": git_commit_sha,
    "firstrade_input_filename": original_input_filename,
    "input_sha256": input_sha256,
    "analysis_date_range": f"{run_info.get('analysis_start')} to {run_info.get('analysis_end')}",
    "price_source": run_info.get("price_source"),
    "latest_tiingo_data_date": run_info.get("api_latest_date"),
    "coverage_status": run_info.get("coverage_status"),
    "tests_result": tests_result,
    "output_local_path": str(latest_run.resolve()),
    "google_drive_output_path": str(drive_run_path),
    "warnings_count": len(warnings),
}
for key, value in completion_summary.items():
    print(f"{key}: {value}")
print("Analysis completed successfully.")